# 2-2. Vector DB & Embedding: 전처리 Markdown 기반 Qdrant 저장

## 학습 목표

- 전처리된 `공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md`를 로드한다.
- Markdown page chunk를 LangChain `Document`로 변환한다.
- `RecursiveCharacterTextSplitter`로 medium chunk를 생성한다.
- `OpenAIEmbeddings`로 문서를 임베딩한다.
- 임베딩 데이터를 Qdrant Server에 영구 저장한다.
- Qdrant Web UI에서 collection, point, payload, metadata를 확인한다.
- 이미 저장된 Qdrant collection을 재사용하여 검색한다.

## 최종 구조

```text
전처리 Markdown 로드
↓
Page chunk 단위 Document 생성
↓
medium chunk 분할
chunk_size=700, chunk_overlap=120
↓
OpenAIEmbeddings 임베딩
↓
Docker 기반 Qdrant Server 저장
↓
Qdrant Web UI에서 metadata 확인
↓
기존 collection 재사용 검색
```

## 0. 사전 작업 1: Python 패키지 설치

### uv 사용 시

```powershell
uv add langchain-qdrant qdrant-client 
```

## 0. 사전 작업 2: Qdrant Server 실행
```text
# 도커 설치
https://www.docker.com/
```

임베딩 데이터를 계속 보관하고 GUI에서 확인하기 위해 **Docker 기반 Qdrant Server**를 사용한다.

아래 명령은 **Windows PowerShell**에서 실행한다.

```powershell
# 1) Qdrant 이미지 다운로드
docker pull qdrant/qdrant

# 2) 영구 저장용 Docker volume 생성
docker volume create qdrant_storage

# 3) 기존 컨테이너가 있다면 정리
docker rm -f qdrant-rag

# 4) Qdrant Server 실행
docker run -d `
  --name qdrant-rag `
  -p 6333:6333 `
  -p 6334:6334 `
  -v qdrant_storage:/qdrant/storage `
  qdrant/qdrant
```

실행 확인:

```powershell
docker ps
```

Web UI 접속:

```text
http://localhost:6333/dashboard
```

> `qdrant_storage` Docker volume을 사용하므로 컨테이너를 재시작해도 collection 데이터가 유지된다.

## 1. 라이브러리 import 및 기본 설정

`MD_PATH`는 현재 노트북 실행 위치를 기준으로 `data` 폴더에 전처리된 Markdown 파일이 있다고 가정한다.

In [20]:
from pathlib import Path
import os, re
from pprint import pprint

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client import models

In [21]:
# .env 탐색: 프로젝트 구조에 따라 위치가 다를 수 있으므로 후보 경로를 순차 확인한다.
load_dotenv(dotenv_path="../../.env", override=True)
load_dotenv(dotenv_path=".env", override=False)

print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

MD_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md")

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "civil_complaint_manual_medium"

EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

# True: collection을 삭제 후 새로 생성한다.
# False: 이미 저장된 collection을 재사용한다.
RECREATE_COLLECTION = True

if not MD_PATH.exists():
    raise FileNotFoundError(f"Markdown 파일을 찾을 수 없습니다: {MD_PATH}")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

OPENAI_API_KEY exists: True


## 2. Qdrant Server 연결 확인

이 셀이 실패하면 Python 코드 문제가 아니라 Qdrant 컨테이너가 실행되지 않았을 가능성이 높다.

In [22]:
client = QdrantClient(url=QDRANT_URL, timeout=10)

try:
    collections = client.get_collections().collections
    print("Qdrant 연결 성공")
    print("현재 collection 목록:")
    for collection in collections:
        print("-", collection.name)
except Exception as e:
    raise RuntimeError(
        "Qdrant Server에 연결할 수 없습니다. "
        "Docker Desktop이 실행 중인지, qdrant-rag 컨테이너가 실행 중인지 확인하세요. "
        "Web UI: http://localhost:6333/dashboard"
    ) 

Qdrant 연결 성공
현재 collection 목록:
- civil_complaint_manual_md_medium


## 3. Markdown page chunk 로드 함수

전처리된 Markdown 파일을 읽어 `## Page n | section | topic` 단위로 분리한다.  
각 page chunk에는 `source`, `page`, `section`, `topic`, `document_type` metadata를 부여한다.

In [23]:
def load_markdown_page_chunks(md_path: Path) -> list[Document]:
    """
    전처리된 RAG page chunk Markdown 파일을 Document 리스트로 읽는다.

    입력 파일 예:
    data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md

    분리 기준:
    ## Page 1 | 섹션 | 주제
    """
    text = md_path.read_text(encoding="utf-8")

    # 문서 전체 제목 제거
    text = re.sub(r"^# .+?\n", "", text, count=1)

    # '## Page n | section | topic' 앞에서 분리
    parts = re.split(r"(?=^## Page\s+\d+\s*\|)", text, flags=re.MULTILINE)

    docs = []

    for part in parts:
        part = part.strip()

        if not part:
            continue

        lines = part.splitlines()
        header = lines[0].strip()

        match = re.match(
            r"^## Page\s+(\d+)\s*\|\s*([^|]+)\s*\|\s*(.+)$",
            header,
        )

        if not match:
            continue

        page = int(match.group(1))
        section = match.group(2).strip()
        topic = match.group(3).strip()

        # HTML comment metadata 제거
        content_lines = [
            line for line in lines[1:]
            if not line.strip().startswith("<!--")
        ]

        content = "\n".join(content_lines).strip()

        if not content:
            continue

        docs.append(
            Document(
                page_content=content,
                metadata={
                    "source": md_path.name,
                    "page": page,
                    "section": section,
                    "topic": topic,
                    "document_type": "preprocessed_markdown",
                    "chunk_type": "page",
                }
            )
        )

    return docs

## 4. medium chunk 분할 함수

전처리된 page chunk를 다시 검색용 medium chunk로 분할한다.

| 항목 | 값 |
|---|---:|
| chunk_size | 700 |
| chunk_overlap | 120 |
| chunk_strategy | medium |

Markdown 문서이므로 separator는 제목, 목록, 줄바꿈을 우선 고려한다.

In [24]:
def make_medium_chunks(
    docs: list[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[Document]:
    """전처리 Markdown 문서에 적합한 medium chunk를 생성한다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n# ", "\n- ", "\n", ". ", " "],
    )

    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata.update(
            {
                "chunk_id": i,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "chunk_strategy": "medium",
                "embedding_model": EMBEDDING_MODEL,
                "collection_name": COLLECTION_NAME,
            }
        )

    return chunks

## 5. chunk 개수 및 샘플 metadata 확인

청크 개수와 샘플 metadata를 확인한다.

In [25]:
pages = load_markdown_page_chunks(MD_PATH)
medium_chunks = make_medium_chunks(pages)

print("page docs:", len(pages))
print("medium chunk 수:", len(medium_chunks))

print("\n[첫 번째 page metadata]")
pprint(pages[0].metadata)

print("\n[첫 번째 medium chunk metadata]")
pprint(medium_chunks[0].metadata)

print("\n[첫 번째 medium chunk preview]")
print(medium_chunks[0].page_content[:700])

page docs: 16
medium chunk 수: 34

[첫 번째 page metadata]
{'chunk_type': 'page',
 'document_type': 'preprocessed_markdown',
 'page': 1,
 'section': '민원응대 관련 기본원칙',
 'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md',
 'topic': '일반 민원응대'}

[첫 번째 medium chunk metadata]
{'chunk_id': 0,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'chunk_type': 'page',
 'collection_name': 'civil_complaint_manual_medium',
 'document_type': 'preprocessed_markdown',
 'embedding_model': 'text-embedding-3-small',
 'page': 1,
 'section': '민원응대 관련 기본원칙',
 'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md',
 'topic': '일반 민원응대'}

[첫 번째 medium chunk preview]
발 간 등 록 번 호
11-1741000-100065-14
## 민원응대


In [26]:
# BASE_DIR = Path.cwd().resolve().parent
# DATA_DIR = BASE_DIR / "data"
# DATA_DIR

## 6. Qdrant collection 생성 또는 재사용

실습에서는 두 가지 모드를 사용한다.

| 모드 | 설정 | 설명 |
|---|---|---|
| 재생성 모드 | `RECREATE_COLLECTION = True` | 기존 collection을 삭제하고 새로 임베딩한다 |
| 재사용 모드 | `RECREATE_COLLECTION = False` | 이미 저장된 collection에 연결만 한다 |

처음 실습할 때는 `True`로 실행한다.  
한 번 저장한 뒤에는 `False`로 바꾸면 임베딩 비용을 다시 쓰지 않고 collection을 재사용할 수 있다.

In [27]:
def collection_exists(client: QdrantClient, collection_name: str) -> bool:
    """Qdrant collection 존재 여부를 확인한다."""
    try:
        return client.collection_exists(collection_name)
    except AttributeError:
        names = [c.name for c in client.get_collections().collections]
        return collection_name in names


exists = collection_exists(client, COLLECTION_NAME)
print(f"collection exists before run: {exists}")

if RECREATE_COLLECTION:
    print("collection을 새로 생성한다. 기존 collection이 있으면 삭제 후 재생성한다.")

    vector_store = QdrantVectorStore.from_documents(
        documents=medium_chunks,
        embedding=embeddings,
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
        force_recreate=True,
    )
else:
    if not exists:
        raise ValueError(
            f"'{COLLECTION_NAME}' collection이 아직 없습니다. "
            "처음 실행할 때는 RECREATE_COLLECTION = True로 설정하세요."
        )

    print("기존 collection을 재사용한다. 새 임베딩을 생성하지 않는다.")

    vector_store = QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        url=QDRANT_URL,
    )

print("vector_store 준비 완료")

collection exists before run: False
collection을 새로 생성한다. 기존 collection이 있으면 삭제 후 재생성한다.
vector_store 준비 완료


## 7. Qdrant collection 정보 확인

저장된 point 수와 collection 설정을 확인한다.

In [28]:
collection_info = client.get_collection(COLLECTION_NAME)

print("collection name:", COLLECTION_NAME)
print("points count:", collection_info.points_count)
print("vectors count:", collection_info.indexed_vectors_count)
print("status:", collection_info.status)
print("\n[collection config]")
pprint(collection_info.config)

collection name: civil_complaint_manual_medium
points count: 34
vectors count: 0
status: green

[collection config]
CollectionConfig(params=CollectionParams(vectors={'': VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_

## 8. Qdrant Web UI에서 metadata 확인하기

브라우저에서 아래 주소로 접속한다.

```text
http://localhost:6333/dashboard
```

확인 순서:

```text
Collections
→ civil_complaint_manual_medium 선택
→ Points 또는 Scroll 메뉴 확인
→ payload 확인
→ page_content와 metadata 확인
```

LangChain이 Qdrant에 저장하는 기본 payload 구조는 대체로 다음과 같다.

```json
{
  "page_content": "청크 원문 텍스트",
  "metadata": {
    "source": "공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md",
    "page": 1,
    "section": "민원응대 관련 기본원칙",
    "topic": "일반 민원응대",
    "document_type": "preprocessed_markdown",
    "chunk_type": "page",
    "chunk_id": 0,
    "chunk_size": 700,
    "chunk_overlap": 120,
    "chunk_strategy": "medium",
    "embedding_model": "text-embedding-3-small"
  }
}
```

## 9. Python으로 Qdrant payload 직접 확인

GUI와 동일하게 Python에서도 Qdrant point의 payload를 확인할 수 있다.  
`with_vectors=False`로 설정하면 긴 벡터값은 출력하지 않고 metadata만 확인할 수 있다.

In [29]:
points, next_page_offset = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

for i, point in enumerate(points, start=1):
    print("=" * 80)
    print(f"[{i}] point id:", point.id)
    print("payload keys:", list(point.payload.keys()))
    print("\nmetadata:")
    pprint(point.payload.get("metadata"))
    print("\npage_content preview:")
    print(point.payload.get("page_content", "")[:500].replace("\n", " "))

print("\nnext_page_offset:", next_page_offset)

[1] point id: 04cefbe6-aa72-477c-9de7-b4ab7548e4f9
payload keys: ['page_content', 'metadata']

metadata:
{'chunk_id': 1,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'chunk_type': 'page',
 'collection_name': 'civil_complaint_manual_medium',
 'document_type': 'preprocessed_markdown',
 'embedding_model': 'text-embedding-3-small',
 'page': 2,
 'section': '민원응대 관련 기본원칙',
 'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md',
 'topic': '폭언'}

page_content preview:
### CONTENTS 4. 온라인 민원 및 문서 민원응대 22 4-1 폭언(욕설, 협박, 성희롱 등) 22 4-2 정당한 사유없는 반복민원 23 민원담당자 회복 및 보호조치 25 질의응답 29 [참고1] 특이민원 의미와 유형 35 [참고2] 위법행위 유형별 적용 법률 36 [참고3] 우울증 또는 스트레스 자가진단법 37 ※ 본 매뉴얼의 민원 응대요령은 민원공무원이 업무 수행 시 참고할 수 있도록 가이드 라인을 제시한 것 입니다. 민원업무 현장에서 다양한 응대 상황이 발생할 수 있으므로 각급 행정기관에서는 본 매뉴얼의 내용을 기관별 상황에 맞게 적절히 활용하시기 바랍니다.
[2] point id: 0624db96-7d44-4858-a8ba-3ef2bb050a33
payload keys: ['page_content', 'metadata']

metadata:
{'chunk_id': 16,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'mediu

## 10. similarity search 테스트

이전 노트북과 동일한 질문으로 검색한다.

In [30]:
question = "전화 중 욕설을 하면 어떻게 대응해야 하나요?"

results = vector_store.similarity_search_with_score(question, k=3)

for i, (doc, score) in enumerate(results, start=1):
    print("=" * 80)
    print(f"[{i}] score: {score}")
    print("metadata:")
    pprint(doc.metadata)
    print("\ncontent preview:")
    print(doc.page_content[:700].replace("\n", " "))

[1] score: 0.51694727
metadata:
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': '402e6af5-85a4-4798-8b0f-37572353b182',
 'chunk_id': 7,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'chunk_type': 'page',
 'collection_name': 'civil_complaint_manual_medium',
 'document_type': 'preprocessed_markdown',
 'embedding_model': 'text-embedding-3-small',
 'page': 6,
 'section': '일반적인 민원 응대요령',
 'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md',
 'topic': '반복전화'}

content preview:
- 위법행위 등에 대한 구체적 입증을 위한 증거자료 확보 STEP - 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 공유하여 개인이 아닌 부서 차원에서 대응 2-1 정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기 핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료 ‣ ~~통화 지속 곤란을 설명한 후 상담종료~~ 정당한 행정처분에 불복하여 반복전화하는 경우 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내 충분한 설명이 이루어졌음에도 반복전화하는 경우 선생님, 말씀하시는 내용에 대해서는 충분히 설명드렸습니다. 다른 분들의 민원처리가 지연되고 있어 통화를 종료하겠습니다. ※ 지속될 경우 행정기관에 부당한 요구를 하

## 11. metadata filter 검색

Qdrant는 payload 기반 필터링이 가능하다.  
아래 예시는 `chunk_strategy == medium`인 point만 검색 대상으로 제한한다.

In [31]:
filtered_results = vector_store.similarity_search_with_score(
    query=question,
    k=3,
    filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.chunk_strategy",
                match=models.MatchValue(value="medium"),
            )
        ]
    ),
)

for i, (doc, score) in enumerate(filtered_results, start=1):
    print("=" * 80)
    print(f"[{i}] score: {score}")
    print("page:", doc.metadata.get("page"))
    print("chunk_id:", doc.metadata.get("chunk_id"))
    print("chunk_strategy:", doc.metadata.get("chunk_strategy"))
    print(doc.page_content[:500].replace("\n", " "))

[1] score: 0.51694727
page: 6
chunk_id: 7
chunk_strategy: medium
- 위법행위 등에 대한 구체적 입증을 위한 증거자료 확보 STEP - 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 공유하여 개인이 아닌 부서 차원에서 대응 2-1 정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기 핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료 ‣ ~~통화 지속 곤란을 설명한 후 상담종료~~ 정당한 행정처분에 불복하여 반복전화하는 경우 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내 충분한 설명이 이루어졌음에도 반복전화하는 경우 선생님, 말씀하시는 내용에 대해서는 충분히 설명드렸습니다. 다른 분들의 민원처리가 지연되고 있어 통화를 종료하겠습니다. ※ 지속될 경우 행정기관에 부당한 요구를 하거나 반복적으로 전화하
[2] score: 0.48214698
page: 7
chunk_id: 8
chunk_strategy: medium
2-2 정당한 사유 없는 장시간 통화 ※ 권장시간 20분 설정의 경우 핵심대응 용건위주 질문유도 및 장시간 전화상담 곤란 안내 2-4 폭언(욕설, 협박, 성희롱 등) 핵심대응 전화 종료 및 증거확보를 통한 후속조치(법적조치 등)로 경각심 부여 ‣ ~~민원전화 수신 시부터 녹음하여 위법행위 증거 확보~~ ‣ ~~15분 이상 통화 장시간 상담 곤란 안내 및 용건 위주 질문유도~~ ※ 민원 신청은 문서가 원칙임을 안내, 지속시 신문고 등 접수 유도 ‣ ~~20분 이상 통화 통화 지속 곤란 안내 후 종료~~ 선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서(또는 대기하고 계시는 다른 민원인이 계셔서) 통화를 종료하오니 양해하여 주시기 바랍니

## 12. Retriever로 변환

RAG 체인에서 사용할 때는 vector store를 retriever로 변환한다.

In [32]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"[{i}] metadata")
    pprint(doc.metadata)
    print("\ncontent preview")
    print(doc.page_content[:500].replace("\n", " "))

[1] metadata
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': '402e6af5-85a4-4798-8b0f-37572353b182',
 'chunk_id': 7,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'chunk_type': 'page',
 'collection_name': 'civil_complaint_manual_medium',
 'document_type': 'preprocessed_markdown',
 'embedding_model': 'text-embedding-3-small',
 'page': 6,
 'section': '일반적인 민원 응대요령',
 'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md',
 'topic': '반복전화'}

content preview
- 위법행위 등에 대한 구체적 입증을 위한 증거자료 확보 STEP - 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 공유하여 개인이 아닌 부서 차원에서 대응 2-1 정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기 핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료 ‣ ~~통화 지속 곤란을 설명한 후 상담종료~~ 정당한 행정처분에 불복하여 반복전화하는 경우 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내 충분한 설명이 이루어졌음에도 반복전화하는 경우 선생님, 말씀하시는 내용에 대해서는 충분히 설명드렸습니다. 다른 분들의 민원처리가 지연되고 있어 통화를 종료하겠습니다. ※ 지속될 경우 행정기관에 부당한 요구를 하거나 반복적으로 전화하
[2] met

## 13. 기존 collection 재사용 전용 코드

다음 수업 또는 다음 실행에서 임베딩을 다시 만들고 싶지 않다면 이 코드만 실행한다.  
단, 처음 한 번은 반드시 collection 생성 셀을 실행해야 한다.

In [33]:
reuse_vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
)

reuse_results = reuse_vector_store.similarity_search(question, k=3)

for i, doc in enumerate(reuse_results, start=1):
    print("=" * 80)
    print(f"[{i}] metadata")
    pprint(doc.metadata)
    print("\ncontent preview")
    print(doc.page_content[:500].replace("\n", " "))

[1] metadata
{'_collection_name': 'civil_complaint_manual_medium',
 '_id': '402e6af5-85a4-4798-8b0f-37572353b182',
 'chunk_id': 7,
 'chunk_overlap': 120,
 'chunk_size': 700,
 'chunk_strategy': 'medium',
 'chunk_type': 'page',
 'collection_name': 'civil_complaint_manual_medium',
 'document_type': 'preprocessed_markdown',
 'embedding_model': 'text-embedding-3-small',
 'page': 6,
 'section': '일반적인 민원 응대요령',
 'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md',
 'topic': '반복전화'}

content preview
- 위법행위 등에 대한 구체적 입증을 위한 증거자료 확보 STEP - 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 공유하여 개인이 아닌 부서 차원에서 대응 2-1 정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기 핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료 ‣ ~~통화 지속 곤란을 설명한 후 상담종료~~ 정당한 행정처분에 불복하여 반복전화하는 경우 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내 충분한 설명이 이루어졌음에도 반복전화하는 경우 선생님, 말씀하시는 내용에 대해서는 충분히 설명드렸습니다. 다른 분들의 민원처리가 지연되고 있어 통화를 종료하겠습니다. ※ 지속될 경우 행정기관에 부당한 요구를 하거나 반복적으로 전화하
[2] met

# 실습 점검 체크리스트

| 점검 항목 | 확인 방법 |
|---|---|
| Qdrant 도커이미지 확인 | `docker images` |
| Qdrant 컨테이너 실행 여부 | `docker ps` |
| Qdrant Web UI 접속 여부 | `http://localhost:6333/dashboard` |
| collection 생성 여부 | Web UI의 Collections 메뉴 |
| point 수 확인 | `client.get_collection(COLLECTION_NAME)` |
| payload 확인 | Web UI 또는 `client.scroll()` |
| metadata 확인 | `source`, `page`, `chunk_id`, `chunk_strategy` |
| 검색 정상 작동 | `similarity_search_with_score()` |
| 재사용 정상 작동 | `from_existing_collection()` |

# 핵심 정리

- 이번 실습에서는 PDF 원본이 아니라 전처리된 Markdown 파일을 사용한다.
- 입력 파일은 `data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md`이다.
- Markdown 파일의 `## Page n | section | topic` 구조를 이용해 page-level `Document`를 생성한다.
- page-level 문서를 다시 `medium chunk`로 분할한다.
- 설정값은 `chunk_size=700`, `chunk_overlap=120`이다.
- `QdrantVectorStore.from_documents()`는 문서를 임베딩하고 Qdrant collection에 저장한다.
- `url="http://localhost:6333"`를 사용하면 Docker 기반 Qdrant Server에 저장된다.
- Docker volume을 연결했기 때문에 컨테이너를 재시작해도 임베딩 데이터가 유지된다.
- Qdrant Web UI에서 collection, point, payload, metadata를 직접 확인할 수 있다.
- 이미 저장된 collection은 `QdrantVectorStore.from_existing_collection()`으로 재사용한다.

## 수업용 설명 문장

> PDF 원본에서 바로 벡터 DB를 만들 수도 있지만, 표와 문서 구조가 중요한 자료는 먼저 Markdown으로 전처리한 뒤 색인하는 방식이 안정적이다.  
> 이번 실습에서는 전처리된 Markdown page chunk를 다시 medium chunk로 분할하고,  
> Qdrant Server에 영구 저장한 뒤 Web UI에서 metadata와 원문을 확인한다.